In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# =============================================================================
# Imports
# =============================================================================
import os
import re
import json
import time
import random
import hashlib
import unicodedata
import urllib.parse
from datetime import datetime, timezone

from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests


In [0]:
# =============================================================================
# Configuração
# =============================================================================

HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")

# URL da listagem, já com o filtro de categoria embutido pelo próprio site.
SITE_URL = (
    "https://www.gov.br/antt/pt-br/assuntos/noticias-defeso-eleitoral"
    "?form.submitted=1&texto=&dt_inicio=&dt_fim=&categoria=infraestrutura-transito-e-transportes"
)

# Base sem query string — usada pra filtrar quais links são "notícia de verdade"
# (um nível abaixo da coleção) e não link de menu/paginação.
COLECAO_BASE_URL = "https://www.gov.br/antt/pt-br/assuntos/noticias-defeso-eleitoral"

SOURCE_ID = "antt_noticias_infraestrutura"
SOURCE_DESCRICAO = "Linked from ANTT — Notícias (Infraestrutura, Trânsito e Transportes)"

PASTA_DESTINO = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/files/{HOJE}/TRANSPORTE"
os.makedirs(PASTA_DESTINO, exist_ok=True)
print(f"[setup] Salvando artefatos em: {PASTA_DESTINO}")

PASTA_MANIFESTOS = "/Volumes/desafio_kinea/research/research_volume/infraestrutura/manifests"
os.makedirs(PASTA_MANIFESTOS, exist_ok=True)
CAMINHO_MANIFESTO = os.path.join(PASTA_MANIFESTOS, f"{SOURCE_ID}_processados.json")

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:126.0) Gecko/20100101 Firefox/126.0",
]

IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

HTTP_TIMEOUT = 30

MIN_CHARS_TEXTO = 200


[setup] Salvando artefatos em: /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-28


In [0]:
# =============================================================================
# Helpers
# =============================================================================

def slugify(texto: str, max_len: int = 80) -> str:
    if not texto:
        return "sem-titulo"
    nfkd = unicodedata.normalize("NFKD", texto)
    ascii_txt = nfkd.encode("ascii", "ignore").decode("ascii")
    ascii_txt = re.sub(r"[^a-zA-Z0-9]+", "-", ascii_txt).strip("-").lower()
    return (ascii_txt[:max_len] or "sem-titulo").strip("-")


def hash_curto(texto: str, n: int = 8) -> str:
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:n]


def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def carregar_manifesto(caminho: str) -> set:
    if not os.path.exists(caminho):
        return set()
    try:
        with open(caminho, "r", encoding="utf-8") as f:
            return set(json.load(f))
    except Exception as e:
        print(f"[manifesto] falha ao carregar ({e}); iniciando vazio.")
        return set()


def salvar_manifesto(caminho: str, urls: set) -> None:
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(sorted(urls), f, ensure_ascii=False, indent=2)


def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer="https://www.gov.br/antt/pt-br")

        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None


In [0]:
# =============================================================================
# Etapa 1 — Extrair a lista de notícias da página 1
# =============================================================================
# Só links um nível abaixo da coleção interessam (ex.: .../noticias-defeso-eleitoral/algum-slug) —
# descarta o link de menu que aponta pra própria raiz da coleção, e qualquer
# link de paginação (que tem "?" na URL).

def extrair_lista_noticias(html: str, url_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    noticias = []
    vistos = set()

    for tag_a in soup.find_all("a", href=True):
        href = tag_a["href"].strip()
        url_absoluta = urllib.parse.urljoin(url_base, href)

        # Só URLs um nível abaixo da coleção, sem query string (exclui paginação
        # e a própria raiz da coleção).
        if not url_absoluta.startswith(COLECAO_BASE_URL + "/"):
            continue
        if "?" in url_absoluta:
            continue
        if url_absoluta in vistos:
            continue
        vistos.add(url_absoluta)

        titulo = tag_a.get_text(" ", strip=True)
        if not titulo:
            continue

        noticias.append({"titulo": titulo, "url": url_absoluta})

    return noticias


In [0]:
# =============================================================================
# Etapa 2 — Limpar o HTML e extrair texto + título + data de publicação
# =============================================================================

TAGS_LIXO = [
    "script", "style", "noscript", "iframe", "svg", "form",
    "nav", "header", "footer", "aside", "button",
]

# Seletores comuns de container de conteúdo em sites Plone (gov.br) — tentados
# em ordem; cai pro texto da página inteira (já sem nav/header/footer) se
# nenhum bater. NÃO TESTADO EM PRODUÇÃO — valide o tamanho do .txt gerado.
SELETORES_CONTEUDO = ["#content-core", "#parent-fieldname-text", "#content", "main"]

PADRAO_DATA_PUBLICACAO = re.compile(r"Publicado em\s*(\d{2}/\d{2}/\d{4})")


def extrair_titulo(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    h1 = soup.find("h1")
    if h1:
        texto = h1.get_text(" ", strip=True)
        if texto:
            return texto
    return None


def extrair_texto(html: str) -> str:
    if not html:
        return ""

    soup = BeautifulSoup(html, "lxml")

    for tag in soup(TAGS_LIXO):
        tag.decompose()

    base = None
    for seletor in SELETORES_CONTEUDO:
        encontrado = soup.select_one(seletor)
        if encontrado and len(encontrado.get_text(strip=True)) > 300:
            base = encontrado
            break

    if base is None:
        base = soup  # fallback: página inteira, já sem nav/header/footer

    texto = base.get_text("\n", strip=True)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()


def extrair_data_publicacao(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    texto_bruto = soup.get_text(" ", strip=True)
    m = PADRAO_DATA_PUBLICACAO.search(texto_bruto)
    if not m:
        return None
    dia, mes, ano = m.group(1).split("/")
    return f"{ano}-{mes}-{dia}"

In [0]:
# =============================================================================
# Etapa 3 — Salvar no Volume
# =============================================================================

def salvar_artefatos(pasta: str, titulo: str, texto: str, metadados: dict) -> tuple[str, str]:
    slug_source = slugify(SOURCE_ID, max_len=40)
    slug_titulo = slugify(titulo, max_len=60) or "sem-titulo"
    sufixo_hash = hash_curto(metadados.get("url") or titulo)

    nome_base = f"{slug_source}_{slug_titulo}_{sufixo_hash}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")

    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_json


In [0]:
# =============================================================================
# Etapa 4 — Pipeline principal
# =============================================================================

def processar_noticia(item: dict) -> Optional[dict]:
    titulo_listagem = item["titulo"]
    url = item["url"]
    print(f"\n  [notícia] {titulo_listagem[:100]}")

    html = baixar_pagina(url)
    if not html:
        print("    -> download falhou; pulando.")
        return None

    titulo = extrair_titulo(html) or titulo_listagem
    texto = extrair_texto(html)

    if not texto or len(texto) < MIN_CHARS_TEXTO:
        print(f"    -> texto muito curto ({len(texto)} chars); pulando.")
        return None

    data_publicacao = extrair_data_publicacao(html)  # busca no HTML bruto, não no texto recortado

    metadados = {
        "source_id": SOURCE_ID,
        "title": titulo,
        "description": SOURCE_DESCRICAO,
        "url": url,
        "date": HOJE,
        "published_at": data_publicacao,
    }

    caminho_txt, caminho_json = salvar_artefatos(PASTA_DESTINO, titulo, texto, metadados)
    print(f"    -> salvo em {caminho_txt}")

    return {"titulo": titulo, "url": url, "caminho_txt": caminho_txt, "caminho_json": caminho_json}


In [0]:
# =============================================================================
# Execução
# =============================================================================

html_listagem = baixar_pagina(SITE_URL)
if not html_listagem:
    raise RuntimeError("Não foi possível baixar a página de listagem de notícias.")

noticias_na_pagina = extrair_lista_noticias(html_listagem, url_base=SITE_URL)
print(f"{len(noticias_na_pagina)} notícias encontradas na página 1.")

ja_processados = carregar_manifesto(CAMINHO_MANIFESTO)
noticias_novas = [n for n in noticias_na_pagina if n["url"] not in ja_processados]
print(f"{len(noticias_novas)} notícias novas (manifesto tinha {len(ja_processados)} conhecidas).")

todos_resultados: list[dict] = []
for item in noticias_novas:
    try:
        resultado = processar_noticia(item)
        if resultado:
            todos_resultados.append(resultado)
            ja_processados.add(item["url"])
    except Exception as e:
        print(f"[ERRO] notícia {item['titulo']!r} falhou: {e}")

salvar_manifesto(CAMINHO_MANIFESTO, ja_processados)

print(f"\n\n=== Fim. {len(todos_resultados)} notícias novas salvas em {PASTA_DESTINO} ===")
print(f"=== Manifesto atualizado: {len(ja_processados)} URLs conhecidas no total ===")


30 notícias encontradas na página 1.
30 notícias novas (manifesto tinha 0 conhecidas).

  [notícia] Duplicação do Contorno Norte de Curitiba alcança 60% de execução
    -> salvo em /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-28/antt-noticias-infraestrutura_duplicacao-do-contorno-norte-de-curitiba-alcanca-60-de-execu_ab408b16.txt

  [notícia] ANTT, Minas Gerais e Belo Horizonte discutem projetos para a Grande BH
    -> salvo em /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-28/antt-noticias-infraestrutura_antt-minas-gerais-e-belo-horizonte-discutem-projetos-para-a_dfc84f27.txt

  [notícia] ANTT reúne contribuições para futura concessão da Malha Sul em audiência pública realizada em Curiti
    -> salvo em /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-28/antt-noticias-infraestrutura_antt-reune-contribuicoes-para-futura-concessao-da-malha-sul_894606e7.txt

  [notícia] ANTT conclui primeira revisão